# Разведочный анализ данных (EDA)
## Датасет: German Credit Risk
Автор: Фех Алексей Александрович

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Настройки визуализации
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Библиотеки успешно загружены')

In [ ]:
import sys
import os

# Добавляем корень проекта в путь для импорта src
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data.loader import load_german_credit

df = load_german_credit()
print(f'Датасет загружен: {df.shape[0]} записей, {df.shape[1]} столбцов')

## 1. Общая информация о датасете

In [ ]:
print(f'Размерность датасета: {df.shape}')
print(f'\nКоличество записей: {df.shape[0]}')
print(f'Количество столбцов: {df.shape[1]}')
print('\n--- Информация о типах данных ---')
df.info()
print('\n--- Описательная статистика (числовые признаки) ---')
df.describe()

## 2. Распределение целевой переменной

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

class_counts = df['class'].value_counts()
class_labels = ['Good (1)', 'Bad (0)']

sns.countplot(data=df, x='class', ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_xticklabels(class_labels)
ax.set_title('Распределение целевой переменной', fontsize=14)
ax.set_xlabel('Класс')
ax.set_ylabel('Количество')

# Добавляем проценты на график
total = len(df)
for i, count in enumerate(class_counts.sort_index().values):
    percentage = count / total * 100
    ax.text(i, count + 10, f'{percentage:.1f}%', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

print(f'\nДоля хороших заёмщиков (class=1): {(df["class"] == 1).mean():.2%}')
print(f'Доля дефолтов (class=0): {(df["class"] == 0).mean():.2%}')

## 3. Анализ числовых признаков

In [ ]:
numerical_features = ['duration', 'credit_amount', 'age',
                      'installment_rate', 'residence_since',
                      'existing_credits', 'people_liable']

fig, axes = plt.subplots(len(numerical_features), 1, figsize=(10, 4 * len(numerical_features)))

for i, col in enumerate(numerical_features):
    ax = axes[i]
    df[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'Распределение: {col}', fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('Частота')
    
    # Добавляем линии среднего и медианы
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val, color='red', linestyle='--', label=f'Среднее: {mean_val:.1f}')
    ax.axvline(median_val, color='green', linestyle='-.', label=f'Медиана: {median_val:.1f}')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Анализ категориальных признаков

In [ ]:
categorical_features = ['checking_account', 'credit_history', 'purpose',
                        'savings_account', 'employment_since', 'personal_status',
                        'other_debtors', 'property', 'other_installment_plans',
                        'housing', 'job', 'telephone', 'foreign_worker']

fig, axes = plt.subplots(len(categorical_features), 1,
                         figsize=(12, 4 * len(categorical_features)))

for i, col in enumerate(categorical_features):
    ax = axes[i]
    value_counts = df[col].value_counts()
    sns.countplot(data=df, y=col, ax=ax, order=value_counts.index, palette='Set2')
    ax.set_title(f'Распределение: {col}', fontsize=12)
    ax.set_xlabel('Количество')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

## 5. Корреляционный анализ

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

corr_matrix = df[numerical_features + ['class']].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Корреляционная матрица числовых признаков', fontsize=14)

plt.tight_layout()
plt.show()

print('Наиболее коррелированные с целевой переменной признаки:')
class_corr = corr_matrix['class'].drop('class').abs().sort_values(ascending=False)
print(class_corr)

## 6. Анализ связи признаков с целевой переменной

In [ ]:
fig, axes = plt.subplots(len(numerical_features), 1,
                         figsize=(10, 4 * len(numerical_features)))

for i, col in enumerate(numerical_features):
    ax = axes[i]
    sns.boxplot(data=df, x='class', y=col, ax=ax, palette=['#e74c3c', '#2ecc71'])
    ax.set_title(f'{col} по классам', fontsize=12)
    ax.set_xticklabels(['Bad (0)', 'Good (1)'])

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

print('Кросс-табуляция и хи-квадрат тест для категориальных признаков:\n')
print(f'{"Признак":<25} {"Хи-квадрат":>12} {"p-value":>12} {"Значимость":>12}')
print('-' * 65)

for col in categorical_features:
    ct = pd.crosstab(df[col], df['class'])
    chi2, p_value, dof, expected = chi2_contingency(ct)
    significance = 'Да' if p_value < 0.05 else 'Нет'
    print(f'{col:<25} {chi2:>12.2f} {p_value:>12.4f} {significance:>12}')

print('\nДетальная кросс-табуляция для checking_account:')
ct_checking = pd.crosstab(df['checking_account'], df['class'], margins=True)
ct_checking_pct = pd.crosstab(df['checking_account'], df['class'], normalize='index')
print(ct_checking)
print('\nДоля классов по категориям checking_account:')
print(ct_checking_pct.round(3))

print('\nДетальная кросс-табуляция для credit_history:')
ct_history = pd.crosstab(df['credit_history'], df['class'], margins=True)
ct_history_pct = pd.crosstab(df['credit_history'], df['class'], normalize='index')
print(ct_history)
print('\nДоля классов по категориям credit_history:')
print(ct_history_pct.round(3))

## 7. Обработка пропусков и выбросов

In [ ]:
# Проверка пропусков
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Пропуски': missing,
    'Процент': missing_pct
})

print('Пропуски в датасете:')
print(missing_df)
print(f'\nОбщее количество пропусков: {df.isnull().sum().sum()}')

# Определение выбросов с помощью метода IQR
print('\n--- Выбросы (метод IQR) ---')
for col in numerical_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    n_outliers = len(outliers)
    pct_outliers = n_outliers / len(df) * 100
    
    print(f'{col:<20} | Выбросов: {n_outliers:>4} ({pct_outliers:>5.1f}%) | '
          f'Границы: [{lower_bound:.1f}, {upper_bound:.1f}]')

## 8. Выводы

**Ключевые результаты EDA:**

1. **Дисбаланс классов**: 70% хороших заёмщиков (class=1) против 30% дефолтов (class=0). Необходимо учитывать при обучении моделей.

2. **Распределения числовых признаков**: Duration и credit_amount имеют правостороннюю асимметрию (right-skewed), что типично для финансовых данных. Возможно, логарифмирование улучшит работу моделей.

3. **Checking account status** — сильный предиктор: заёмщики без расчётного счёта имеют значительно меньший риск дефолта.

4. **Credit history** существенно влияет на вероятность дефолта: заёмщики с хорошей кредитной историей реже допускают дефолт.

5. **Возраст**: большинство заёмщиков в диапазоне 20-40 лет, распределение также скошено вправо.

6. **Пропуски**: в датасете отсутствуют пропущенные значения, что упрощает предобработку.

7. **Выбросы**: в признаке credit_amount обнаружены выбросы по методу IQR, однако они могут быть информативными (крупные кредиты = высокий риск), поэтому удалять их не рекомендуется.